In [1]:
import numpy as np
import pandas as pd

In [2]:
# data_ids = [361260, 361254, 361259, 361253, 361243, 361242]
data_ids = [361254, 361259, 361253, 361242]
n_ests = [50, 100, 500, 1000]
min_samples_leafs = [1, 5, 10]
max_features = [0.1, 0.33, "1.0"]

In [3]:
# for each data_id, load the result and save as a df
dfs = []
for data_id in data_ids:
    # get number of samples in the data_id by reading X csv
    X = np.loadtxt(f"data/{data_id}/X.csv", delimiter=",")
    n_samples = 1000
    n_features = X.shape[1]
    for n_est in n_ests:
        for min_samples_leaf in min_samples_leafs:
            for max_feature in max_features:
                # create the directory if it doesn't exist
                dir_path = f"results/rf/{data_id}/n_estimators_{n_est}/min_samples_leaf_{min_samples_leaf}/max_features_{max_feature}"
                results_path = f"{dir_path}/runtime_results.csv"
                results_df = pd.read_csv(results_path)
                # divide every col in df except 'data_id' by n_samples
                for col in results_df.columns:
                    if col != 'data_id':
                        results_df[col] = results_df[col] / n_samples
                # add columns for n_estimators, min_samples_leaf, max_features
                results_df['n_estimators'] = n_est
                results_df['min_samples_leaf'] = min_samples_leaf
                results_df['max_features'] = max_feature
                results_df['num_features'] = n_features
                dfs.append(results_df)
df = pd.concat(dfs, ignore_index=True)

In [4]:
df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])

,data_id,rf_fitting_time,rf_plus_fitting_time,shap_rf_explainer_time,shap_rf_values_time,lime_rf_time,local_mdi_time,lmdi_plus_rf_explainer_time,lmdi_plus_rf_values_time,n_estimators,min_samples_leaf,max_features,num_features
118,361242,0.001498,0.047808,0.000030,0.031717,0.186433,0.001082,0.000006,0.043541,100,1,0.33,81
82,361253,0.001102,0.027166,0.000032,0.033963,0.138509,0.001114,0.000006,0.022529,100,1,0.33,48
10,361254,0.000665,0.022531,0.000025,0.031032,0.126291,0.001288,0.000009,0.015159,100,1,0.33,21
46,361259,0.000813,0.045874,0.000033,0.030291,0.114102,0.001223,0.000006,0.014599,100,1,0.33,32
121,361242,0.001003,0.007108,0.000004,0.003114,0.188586,0.000834,0.000006,0.034292,100,5,0.33,81
85,361253,0.000853,0.008118,0.000010,0.003883,0.168553,0.000920,0.000006,0.020327,100,5,0.33,48
13,361254,0.000395,0.005046,0.000012,0.002404,0.073060,0.000779,0.000005,0.003176,100,5,0.33,21
49,361259,0.000571,0.005350,0.000012,0.002810,0.095760,0.000815,0.000004,0.005758,100,5,0.33,32
124,361242,0.000844,0.005588,0.000008,0.001079,0.167529,0.000632,0.000005,0.016736,100,10,0.33,81
88,361253,0.000723,0.007096,0.000010,0.001369,0.179425,0.000804,0.000006,0.010872,100,10,0.33,48


In [5]:
display_df = df[(df['max_features'] == 0.33) & (df['n_estimators'] == 100)].sort_values(by=['min_samples_leaf', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, rf_plus_fitting_time + lmdi_plus_values_time, lime_time, shap_values_time, local_mdi_time
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'rf_plus_fitting_time', 'lmdi_plus_rf_values_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_rf_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_rf_values_time'], inplace=True)
display_df = display_df[['data_id', 'num_features', 'min_samples_leaf', 'lmdi_plus_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lmdi_plus_time': 'LMDI+',
    'lime_rf_time': 'LIME',
    'shap_rf_values_time': 'TreeSHAP',
    'local_mdi_time': 'Local MDI'
})

# sort display_df by min samples leaf increasing and then number of features increasing
display_df = display_df.sort_values(by=['Min. Samples per Leaf', '# of Features'])

# round to fourth decimal place
display_df = display_df.round(4)

In [6]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   Min. Samples per Leaf |   LMDI+ |   LIME |   TreeSHAP |   Local MDI |
|-----------------:|----------------:|------------------------:|--------:|-------:|-----------:|------------:|
|           361254 |              21 |                       1 |  0.0377 | 0.1263 |     0.031  |      0.0013 |
|           361259 |              32 |                       1 |  0.0605 | 0.1141 |     0.0303 |      0.0012 |
|           361253 |              48 |                       1 |  0.0497 | 0.1385 |     0.034  |      0.0011 |
|           361242 |              81 |                       1 |  0.0913 | 0.1864 |     0.0317 |      0.0011 |
|           361254 |              21 |                       5 |  0.0082 | 0.0731 |     0.0024 |      0.0008 |
|           361259 |              32 |                       5 |  0.0111 | 0.0958 |     0.0028 |      0.0008 |
|           361253 |              48 |                       5 |  0.0284 | 0.1686 |     0.0039 |      0.0009 |
|

In [7]:
# get display_df in latex format, still only use 4 decimal places
latex_df = display_df.to_latex(index=False, float_format="%.4f")
print(latex_df)

\begin{tabular}{rrrrrrr}
\toprule
OpenML Data ID & # of Features & Min. Samples per Leaf & LMDI+ & LIME & TreeSHAP & Local MDI \\
\midrule
361254 & 21 & 1 & 0.0377 & 0.1263 & 0.0310 & 0.0013 \\
361259 & 32 & 1 & 0.0605 & 0.1141 & 0.0303 & 0.0012 \\
361253 & 48 & 1 & 0.0497 & 0.1385 & 0.0340 & 0.0011 \\
361242 & 81 & 1 & 0.0913 & 0.1864 & 0.0317 & 0.0011 \\
361254 & 21 & 5 & 0.0082 & 0.0731 & 0.0024 & 0.0008 \\
361259 & 32 & 5 & 0.0111 & 0.0958 & 0.0028 & 0.0008 \\
361253 & 48 & 5 & 0.0284 & 0.1686 & 0.0039 & 0.0009 \\
361242 & 81 & 5 & 0.0414 & 0.1886 & 0.0031 & 0.0008 \\
361254 & 21 & 10 & 0.0105 & 0.1052 & 0.0012 & 0.0009 \\
361259 & 32 & 10 & 0.0101 & 0.0983 & 0.0012 & 0.0008 \\
361253 & 48 & 10 & 0.0180 & 0.1794 & 0.0014 & 0.0008 \\
361242 & 81 & 10 & 0.0223 & 0.1675 & 0.0011 & 0.0006 \\
\bottomrule
\end{tabular}



In [8]:
df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5)].sort_values(by=['n_estimators', 'data_id'])

,data_id,rf_fitting_time,rf_plus_fitting_time,shap_rf_explainer_time,shap_rf_values_time,lime_rf_time,local_mdi_time,lmdi_plus_rf_explainer_time,lmdi_plus_rf_values_time,n_estimators,min_samples_leaf,max_features,num_features
112,361242,0.000532,0.006824,0.000005,0.001733,0.215546,0.000483,0.000003,0.015815,50,5,0.33,81
76,361253,0.000476,0.005429,0.000007,0.002090,0.156516,0.000562,0.000004,0.008764,50,5,0.33,48
4,361254,0.000180,0.003806,0.000004,0.001255,0.055506,0.000423,0.000002,0.001566,50,5,0.33,21
40,361259,0.000366,0.005601,0.000005,0.001970,0.117854,0.000586,0.000004,0.004816,50,5,0.33,32
121,361242,0.001003,0.007108,0.000004,0.003114,0.188586,0.000834,0.000006,0.034292,100,5,0.33,81
85,361253,0.000853,0.008118,0.000010,0.003883,0.168553,0.000920,0.000006,0.020327,100,5,0.33,48
13,361254,0.000395,0.005046,0.000012,0.002404,0.073060,0.000779,0.000005,0.003176,100,5,0.33,21
49,361259,0.000571,0.005350,0.000012,0.002810,0.095760,0.000815,0.000004,0.005758,100,5,0.33,32
130,361242,0.005007,0.039496,0.000050,0.014647,0.300730,0.003435,0.000029,0.116405,500,5,0.33,81
94,361253,0.003813,0.034524,0.000052,0.015624,0.261150,0.003626,0.000028,0.054999,500,5,0.33,48


In [9]:
display_df

,OpenML Data ID,# of Features,Min. Samples per Leaf,LMDI+,LIME,TreeSHAP,Local MDI
10,361254,21,1,0.0377,0.1263,0.0310,0.0013
46,361259,32,1,0.0605,0.1141,0.0303,0.0012
82,361253,48,1,0.0497,0.1385,0.0340,0.0011
118,361242,81,1,0.0913,0.1864,0.0317,0.0011
13,361254,21,5,0.0082,0.0731,0.0024,0.0008
49,361259,32,5,0.0111,0.0958,0.0028,0.0008
85,361253,48,5,0.0284,0.1686,0.0039,0.0009
121,361242,81,5,0.0414,0.1886,0.0031,0.0008
16,361254,21,10,0.0105,0.1052,0.0012,0.0009
52,361259,32,10,0.0101,0.0983,0.0012,0.0008


In [10]:
display_df = df[(df['max_features'] == 0.33) & (df['min_samples_leaf'] == 5) & (df['n_estimators'] != 50)].sort_values(by=['n_estimators', 'data_id'])
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, rf_plus_fitting_time + lmdi_plus_values_time, lime_time, shap_values_time, local_mdi_time
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'rf_plus_fitting_time', 'lmdi_plus_rf_values_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_rf_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_rf_values_time'], inplace=True)
display_df = display_df[['data_id', 'num_features', 'n_estimators', 'lmdi_plus_time', 'lime_rf_time', 'shap_rf_values_time', 'local_mdi_time']]
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lmdi_plus_time': 'LMDI+',
    'lime_rf_time': 'LIME',
    'shap_rf_values_time': 'TreeSHAP',
    'local_mdi_time': 'Local MDI'
})

# sort display_df by min samples leaf increasing and then number of features increasing
display_df = display_df.sort_values(by=['# of Estimators', '# of Features'])

# round to fourth decimal place
display_df = display_df.round(4)

In [11]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   # of Estimators |   LMDI+ |   LIME |   TreeSHAP |   Local MDI |
|-----------------:|----------------:|------------------:|--------:|-------:|-----------:|------------:|
|           361254 |              21 |               100 |  0.0082 | 0.0731 |     0.0024 |      0.0008 |
|           361259 |              32 |               100 |  0.0111 | 0.0958 |     0.0028 |      0.0008 |
|           361253 |              48 |               100 |  0.0284 | 0.1686 |     0.0039 |      0.0009 |
|           361242 |              81 |               100 |  0.0414 | 0.1886 |     0.0031 |      0.0008 |
|           361254 |              21 |               500 |  0.0474 | 0.2007 |     0.0118 |      0.0037 |
|           361259 |              32 |               500 |  0.088  | 0.2927 |     0.0181 |      0.0046 |
|           361253 |              48 |               500 |  0.0895 | 0.2611 |     0.0156 |      0.0036 |
|           361242 |              81 |               50

In [12]:
# get display_df in latex format
latex_df = display_df.to_latex(index=False, float_format="%.4f")
print(latex_df)

\begin{tabular}{rrrrrrr}
\toprule
OpenML Data ID & # of Features & # of Estimators & LMDI+ & LIME & TreeSHAP & Local MDI \\
\midrule
361254 & 21 & 100 & 0.0082 & 0.0731 & 0.0024 & 0.0008 \\
361259 & 32 & 100 & 0.0111 & 0.0958 & 0.0028 & 0.0008 \\
361253 & 48 & 100 & 0.0284 & 0.1686 & 0.0039 & 0.0009 \\
361242 & 81 & 100 & 0.0414 & 0.1886 & 0.0031 & 0.0008 \\
361254 & 21 & 500 & 0.0474 & 0.2007 & 0.0118 & 0.0037 \\
361259 & 32 & 500 & 0.0880 & 0.2927 & 0.0181 & 0.0046 \\
361253 & 48 & 500 & 0.0895 & 0.2611 & 0.0156 & 0.0036 \\
361242 & 81 & 500 & 0.1559 & 0.3007 & 0.0146 & 0.0034 \\
361254 & 21 & 1000 & 0.2232 & 0.3737 & 0.0281 & 0.0084 \\
361259 & 32 & 1000 & 0.2056 & 0.3913 & 0.0296 & 0.0086 \\
361253 & 48 & 1000 & 0.3144 & 0.5009 & 0.0417 & 0.0088 \\
361242 & 81 & 1000 & 0.5098 & 0.5033 & 0.0332 & 0.0077 \\
\bottomrule
\end{tabular}

